# 第 5 周 · 周六：TopN / 复购率 / 漏斗——三大经典业务 SQL

> 难度：⭐⭐⭐ | 今日目标：掌握三道面试和工作中最常见的分析SQL，覆盖70%的日常工作
> 配对课程：01python_learn W5D6（综合练习/API数据获取）
> 前置依赖：本周二窗口函数 + 本周四JOIN + 本周五CTE
> 扩展阅读：02analyst_thinking/week_01/day_05（日期函数与时间序列）
> 面试地位：大厂SQL面试必考，三题覆盖90%以上的业务SQL面试题

---

## 一、核心概念——三道题覆盖70%的日常工作

### 1.1 TopN：每组取前N名

**业务场景**：每个品类销售额Top10的商品、每个城市消费Top5的用户、每天GMV Top3的品类。

**核心模式**：
```sql
SELECT * FROM (
  SELECT *,
    ROW_NUMBER() OVER(PARTITION BY group_col ORDER BY value_col DESC) AS rn
  FROM table
) WHERE rn <= N;
```

**关键选择——用哪个排名函数？**

| 函数 | 并列处理 | 风险 | 适用 |
|------|---------|------|------|
| ROW_NUMBER | 强制唯一编号 | 并列的数据只有1个上榜 | 预算固定的场景 |
| RANK | 并列占用多个位置 | 可能超过N个人 | 比赛排名、公平优先 |
| DENSE_RANK | 并列但不跳号 | 可能超过N个人 | 等级划分、分组排名 |

### 1.2 复购率：买了又买的用户比例

**业务场景**：评估用户忠诚度、品类粘性、渠道质量。

**核心公式**：
```
复购率 = 购买≥2次的用户数 / 总购买用户数
```

**核心模式**：
```sql
WITH user_orders AS (
    SELECT user_id, COUNT(DISTINCT order_id) AS order_cnt
    FROM orders
    WHERE dt BETWEEN start AND end
    GROUP BY user_id
)
SELECT
    COUNT(*) AS total_users,
    COUNT(CASE WHEN order_cnt >= 2 THEN 1 END) AS repeat_users,
    ROUND(COUNT(CASE WHEN order_cnt >= 2 THEN 1 END) * 100.0 / COUNT(*), 1) AS repurchase_rate
FROM user_orders;
```

**复购率的变体**：
- 按品类：每个品类的复购率
- 按时间窗：30天内复购率、90天内复购率
- 按渠道：不同获客渠道的用户复购率
- 跨品类复购：买了A品类的用户又买了B品类的比例

### 1.3 漏斗：每一步还剩多少人

**业务场景**：注册转化漏斗、下单支付漏斗、活动参与漏斗。

**核心模式（简单版——只看整体）**：
```sql
SELECT
    COUNT(DISTINCT user_id) AS step0_total,
    COUNT(DISTINCT CASE WHEN event='view' THEN user_id END) AS step1_view,
    COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END) AS step2_cart,
    COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END) AS step3_pay
FROM events
WHERE dt = '2026-01-15';
```

**进阶模式（追踪同一会话内的漏斗）**：
- 需要加session_id来确保是"同一次"行为序列
- 需要加时间窗口（如在浏览后30分钟内加购才算）

### 1.4 三题之间的关系

这三道题有一个共同的底层模式：**先聚合到需要的粒度，再计算指标**。

```
原始数据 → 聚合/窗口到分析粒度 → 指标计算
```

- TopN：聚合到组粒度 → 组内排名 → 取前N
- 复购率：聚合到用户粒度 → 计数分类 → 比例
- 漏斗：聚合到用户/会话粒度 → 逐层计数 → 转化率

> **今天最重要的认知**：TopN、复购率、漏斗——三道题覆盖了分析师日常SQL工作的70%。它们的共同模式："先GROUP BY到分析粒度，再用窗口函数或CASE WHEN计算目标指标"。掌握了这三道题，你就掌握了SQL分析80%的实战能力。

---

## 二、知识点详细拆解

### 2.1 TopN的四种变体

**变体1：每组严格N个（ROW_NUMBER）**
```sql
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY category ORDER BY amount DESC) AS rn
    FROM orders
) WHERE rn <= 3;
```

**变体2：每组取前N%，不固定N（PERCENT_RANK）**
```sql
SELECT * FROM (
    SELECT *, PERCENT_RANK() OVER(PARTITION BY category ORDER BY amount DESC) AS pct
    FROM orders
) WHERE pct <= 0.1;  -- 前10%
```

**变体3：分组后聚合再TopN（最常见）**
```sql
-- 每个品类消费Top3的用户
SELECT * FROM (
    SELECT category, user_id, SUM(amount) AS total,
        ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rn
    FROM orders
    GROUP BY category, user_id
) WHERE rn <= 3;
```

**变体4：多层TopN（每个城市内每个品类Top3）**
```sql
SELECT * FROM (
    SELECT city, category, user_id, SUM(amount) AS total,
        ROW_NUMBER() OVER(PARTITION BY city, category ORDER BY SUM(amount) DESC) AS rn
    FROM orders
    GROUP BY city, category, user_id
) WHERE rn <= 3;
```

### 2.2 复购率的六个变体

| 变体 | SQL关键差异 | 业务含义 |
|------|-----------|--------|
| 整体复购率 | COUNT(*) | 全平台用户粘性 |
| 品类复购率 | GROUP BY category | 不同品类的粘性差异 |
| 时间窗复购率 | WHERE dt BETWEEN ... | 不同时间窗的复购行为 |
| 渠道复购率 | GROUP BY channel | 不同渠道的用户质量 |
| 跨品类复购率 | COUNT(DISTINCT category) >= 2 | 用户的品类多样性 |
| 复购周期 | AVG(复购间隔) | 多久回来再买 |

**复购周期分析**：
```sql
WITH ordered AS (
    SELECT user_id, order_date,
        LAG(order_date) OVER(PARTITION BY user_id ORDER BY order_date) AS prev_date
    FROM orders
),
intervals AS (
    SELECT user_id,
        JULIANDAY(order_date) - JULIANDAY(prev_date) AS days_between
    FROM ordered
    WHERE prev_date IS NOT NULL
)
SELECT
    ROUND(AVG(days_between), 1) AS avg_repurchase_days,
    ROUND(MEDIAN(days_between), 1) AS median_repurchase_days
FROM intervals;
```

### 2.3 漏斗的三种精度

**精度1：天级整体漏斗**（最简单，但最不精确）
- 条件：同一天内发生过各个步骤
- 问题：不同步骤可能发生在不同会话中

**精度2：会话级漏斗**（需要Sessionization，精确度中等）
- 条件：同一会话内的步骤序列
- SQL：用session_id关联各步骤

**精度3：事件级顺序漏斗**（最精确）
- 条件：严格按时间顺序的步骤序列
- SQL：用窗口函数追踪步骤顺序

```sql
-- 事件级漏斗：用户必须严格按 view→cart→pay 顺序
WITH ordered_events AS (
    SELECT user_id, event, event_time,
        LAG(event) OVER(PARTITION BY user_id ORDER BY event_time) AS prev_event
    FROM events
    WHERE dt = '2026-01-15'
)
SELECT
    COUNT(DISTINCT CASE WHEN event='view' THEN user_id END) AS viewed,
    COUNT(DISTINCT CASE WHEN event='add_cart' AND prev_event='view' THEN user_id END) AS carted,
    COUNT(DISTINCT CASE WHEN event='purchase' AND prev_event='add_cart' THEN user_id END) AS purchased
FROM ordered_events;
```

### 2.4 漏斗转化率的计算

**为什么重要**：只看绝对人数不够，转化率才是核心指标。

| 转化关系 | 公式 | 典型值 |
|---------|------|--------|
| 浏览→加购 | 加购人数/浏览人数 | 10-20% |
| 加购→下单 | 下单人数/加购人数 | 30-50% |
| 下单→支付 | 支付人数/下单人数 | 70-90% |
| 浏览→支付（全链路） | 支付人数/浏览人数 | 2-5% |

### 2.5 TopN + 复购率 + 漏斗的组合使用

**实际业务中，这三道题经常组合出现**：

需求："找出Top10品类中复购率最高的3个品类，并给出这些品类的浏览→加购→下单→支付漏斗。"

这种需求需要：TopN（找Top10品类）→ 复购率（筛选高复购）→ 漏斗（分析转化路径）。

用CTE把每一步分开，逻辑清晰，代码可维护。

### 2.6 漏斗分析的陷阱

| 陷阱 | 描述 | 解决方案 |
|------|------|---------|
| 跨天漏斗 | 用户第一天浏览，第二天购买 | 扩大时间窗口或在会话级别追踪 |
| 多渠道重叠 | 用户用App浏览，PC购买 | 需要打通用户标识 |
| 步骤回退 | 用户加购后返回浏览再购买 | 只追踪首次到达或按顺序匹配 |
| 时间窗口定义 | "浏览后多久内购买"的定义不同 | 明确业务口径 |

---

## 三、动手练习

用Python的sqlite3模拟完整的三道题练习。

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 订单表
n = 300
df = pd.DataFrame({
    'user_id': np.random.choice(range(1, 51), n),
    'category': np.random.choice(['服装', '数码', '食品', '美妆'], n),
    'amount': np.round(np.random.lognormal(4, 0.8, n), 2),
    'order_date': np.random.choice(pd.date_range('2026-01-01', '2026-01-31'), n)
})
df['order_date'] = df['order_date'].astype(str)
df.to_sql('orders', conn, index=False)

# 事件表（漏斗用）
events = pd.DataFrame({
    'user_id': np.random.choice(range(1, 101), 500),
    'event': np.random.choice(['view', 'add_cart', 'purchase'], 500, p=[0.6, 0.25, 0.15]),
    'dt': '2026-01-15'
})
events.to_sql('events', conn, index=False)

print(f"订单数据: {n} 行")
print(f"事件数据: 500 行（100个用户模拟漏斗）")
print()

### 练习1：TopN——每个品类消费Top3用户

In [ ]:
print("=" * 60)
print("练习1：每个品类消费金额Top3用户")
print("=" * 60)

q1 = """SELECT category, user_id,
    ROUND(total, 1) AS total_amount,
    order_cnt, rn
FROM (
    SELECT category, user_id,
        SUM(amount) AS total,
        COUNT(*) AS order_cnt,
        ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rn
    FROM orders
    GROUP BY category, user_id
)
WHERE rn <= 3
ORDER BY category, rn"""
print(pd.read_sql(q1, conn).to_string())
print()
print("如果用RANK替代ROW_NUMBER——可能出现超过3人的情况")
print("如果预算有限（只发3张券），必须用ROW_NUMBER")

### 练习2：复购率——整体 + 按品类拆解

In [ ]:
print("=" * 60)
print("练习2A：整体复购率")
print("=" * 60)

q2a = """WITH user_orders AS (
    SELECT user_id, COUNT(*) AS order_cnt
    FROM orders
    GROUP BY user_id
)
SELECT COUNT(*) AS total_users,
    SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) AS repeat_users,
    ROUND(SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS repurchase_rate
FROM user_orders"""
print(pd.read_sql(q2a, conn))

print()
print("=" * 60)
print("练习2B：按品类复购率")
print("=" * 60)

q2b = """WITH user_category_orders AS (
    SELECT user_id, category, COUNT(*) AS order_cnt
    FROM orders
    GROUP BY user_id, category
)
SELECT category,
    COUNT(*) AS total_users,
    SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) AS repeat_users,
    ROUND(SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS repurchase_rate
FROM user_category_orders
GROUP BY category
ORDER BY repurchase_rate DESC"""
print(pd.read_sql(q2b, conn))
print()
print("食品和服装的复购率通常高于数码——用户可能更频繁地买食品")

### 练习3：漏斗——浏览→加购→购买

In [ ]:
print("=" * 60)
print("练习3A：简单漏斗（只看是否发生过该事件）")
print("=" * 60)

q3a = """SELECT
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN event='view' THEN user_id END) AS viewed,
    COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END) AS carted,
    COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END) AS purchased
FROM events"""
result3a = pd.read_sql(q3a, conn)
print(result3a)

print()
print("=" * 60)
print("练习3B：带转化率的漏斗")
print("=" * 60)

q3b = """WITH funnel AS (
    SELECT
        COUNT(DISTINCT user_id) AS total_users,
        COUNT(DISTINCT CASE WHEN event='view' THEN user_id END) AS viewed,
        COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END) AS carted,
        COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END) AS purchased
    FROM events
)
SELECT
    'view' AS step, viewed AS users, 100.0 AS rate
FROM funnel
UNION ALL
SELECT
    'cart', carted, ROUND(carted * 100.0 / viewed, 1)
FROM funnel
UNION ALL
SELECT
    'purchase', purchased, ROUND(purchased * 100.0 / carted, 1)
FROM funnel"""
print(pd.read_sql(q3b, conn))

print()
print("观察：每一步的转化率如何？哪个环节流失最严重？")

### 练习4：跨品类复购——买了A又来买B的用户

In [ ]:
print("=" * 60)
print("练习4：跨品类复购分析（买了A又买了B的用户比例）")
print("=" * 60)

q4 = """WITH user_categories AS (
    SELECT user_id, COUNT(DISTINCT category) AS category_cnt
    FROM orders
    GROUP BY user_id
)
SELECT
    COUNT(*) AS total_users,
    SUM(CASE WHEN category_cnt >= 2 THEN 1 ELSE 0 END) AS cross_category_users,
    ROUND(SUM(CASE WHEN category_cnt >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS cross_rate
FROM user_categories"""
print(pd.read_sql(q4, conn))
print()
print("跨品类复购率越高，说明平台生态越好，用户粘性越强")
print("如果用户只在一个品类消费——平台对其价值单一，流失风险高")

### 练习5：挑战——综合三题的业务场景

运营需要做一次"用户健康度月报"，需要：
1. 每个品类消费Top5用户的消费金额和排名
2. 整体复购率和每个品类的复购率
3. 如果把"至少浏览过"→"加购过"→"购买过"作为漏斗，转化率如何？

请用CTE组织以上三个分析，写在一个查询里（或用三个独立查询）。

In [ ]:
print("=" * 60)
print("挑战练习：综合月报")
print("=" * 60)

print()
print("PART 1: 每个品类消费Top5用户")
q5a = """SELECT category, user_id,
    ROUND(total, 1) AS total_amount, order_cnt, rn
FROM (
    SELECT category, user_id,
        SUM(amount) AS total, COUNT(*) AS order_cnt,
        ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rn
    FROM orders GROUP BY category, user_id
)
WHERE rn <= 5
ORDER BY category, rn"""
print(pd.read_sql(q5a, conn).to_string())

print()
print("PART 2: 复购率（整体+按品类）")
q5b = """WITH uo AS (
    SELECT user_id, COUNT(*) AS cnt FROM orders GROUP BY user_id
)
SELECT '整体' AS category,
    COUNT(*) AS users,
    SUM(CASE WHEN cnt>=2 THEN 1 ELSE 0 END) AS repeat_u,
    ROUND(SUM(CASE WHEN cnt>=2 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS rate
FROM uo"""
print(pd.read_sql(q5b, conn))

print()
print("PART 3: 漏斗")
q5c = """SELECT
    COUNT(DISTINCT CASE WHEN event='view' THEN user_id END) AS viewed,
    COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END) AS carted,
    COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END) AS purchased,
    ROUND(COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END)*100.0/
        NULLIF(COUNT(DISTINCT CASE WHEN event='view' THEN user_id END),0),1) AS view_to_cart,
    ROUND(COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END)*100.0/
        NULLIF(COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END),0),1) AS cart_to_pay
FROM events"""
print(pd.read_sql(q5c, conn))
print()
print("这份月报现在可以直接发给运营——数字+解读一条龙")

---

## 四、业务场景应用

### 场景1：运营说"帮我拉一下各品类Top10商品"——你的追问

你："是按GMV排名还是按销量排名？如果第10名和第11名金额一样怎么办？"

运营："那就都取。"

你的判断：如果用ROW_NUMBER，并列的会被丢掉一个。应该用RANK——并列都算。

### 场景2：某品类复购率从25%降到18%

**排查**：
1. 先按时间看趋势：是突然下降还是逐渐下降？
2. 按渠道拆：是新用户的复购降了还是老用户的复购降了？
3. 看该品类的价格变化、竞品动作、是否改版

**SQL**：按周计算复购率趋势
```sql
WITH weekly_orders AS (
    SELECT strftime('%Y-W%W', order_date) AS week,
           user_id, COUNT(*) AS order_cnt
    FROM orders
    GROUP BY week, user_id
)
SELECT week,
       COUNT(*) AS users,
       SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) AS repeat_users,
       ROUND(SUM(CASE WHEN order_cnt >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS rate
FROM weekly_orders
GROUP BY week
ORDER BY week DESC;
```

### 场景3：漏斗某一步骤转化率暴跌

昨天"加购→下单"的转化率从35%降到15%。

**排查**：
1. 看是不是后端bug——下单接口是否正常
2. 看是不是前端改版——加购按钮改了位置，用户误触
3. 看是不是数据bug——加购事件重复上报

### 场景4：CEO问"我们的用户质量怎么样？"

用复购率+漏斗+TopN三个维度回答：
- 复购率：30天内复购率38%，同比提升3ppt
- 漏斗：浏览→支付转化率4.2%，行业平均3.5%
- TopN品类中，食品的复购率最高（45%），数码最低（22%）

三个指标一起看，才能给出全面的"用户质量"评估。

---

## 五、分级检验清单

### 初级检验（必须能独立完成）
- [ ] 能写出每组取TopN的标准SQL
- [ ] 能写出整体复购率的SQL
- [ ] 能写出简单漏斗的SQL
- [ ] 知道ROW_NUMBER和RANK在TopN中的区别

**答案提示**：
- TopN：ROW_NUMBER + PARTITION BY + WHERE rn<=N
- 复购率：COUNT(CASE WHEN order_cnt>=2) / COUNT(*)
- 漏斗：COUNT(DISTINCT CASE WHEN event=X THEN user_id END)

### 中级检验（能解释给同事听）
- [ ] 能解释不同排名函数在TopN中的业务影响（预算控制）
- [ ] 能按品类/渠道/时间拆解复购率
- [ ] 能区分"天级漏斗""会话级漏斗""事件级漏斗"
- [ ] 能说出漏斗分析的两个核心陷阱

**答案提示**：
- ROW_NUMBER严格N人（控制预算），RANK可能超N人
- 复购率拆解=在GROUP BY里加对应的维度列
- 精度递增：天级<会话级<事件级
- 陷阱：跨天漏斗和多渠道重叠

### 高级检验（能举一反三）
- [ ] 能组合使用TopN+复购率+漏斗完成综合分析
- [ ] 能设计"事件级顺序漏斗"SQL
- [ ] 能从复购率异常追溯到渠道/品类/用户群
- [ ] 能向CEO汇报三指标的用户质量全景

**答案提示**：
- 综合分析：用CTE组织分别计算，最后汇总
- 顺序漏斗：用LAG+条件判断追踪事件顺序
- 追溯异常：按维度拆解，看哪个维度变化最大
- 全景汇报：复购率+漏斗+TopN一起看

---

## 六、跨模块连接

| 方向 | 关联内容 | 说明 |
|------|---------|------|
| **← 本周二** | 窗口函数 | TopN依赖ROW_NUMBER窗口函数 |
| **← 本周四** | JOIN | 跨品类复购需要自连接 |
| **← 本周五** | CTE | 复杂综合查询用CTE分层 |
| **→ 第6周** | 多维拆解分析 | TopN+复购+漏斗的多维组合 |
| **→ 第7周** | AB实验分析 | 漏斗是实验中衡量转化的核心指标 |
| **扩展阅读** | 02analyst_thinking W1D5 | 时间序列分析（含漏斗进阶） |

### 横向连接：三大经典SQL在其他领域的应用

| 领域 | TopN | 复购率 | 漏斗 |
|------|------|--------|------|
| 电商 | 各品类Top10商品 | 用户复购率 | 浏览→下单→支付 |
| 游戏 | 各服务器战力Top100 | 玩家回流率 | 下载→注册→首充 |
| 内容 | 各频道Top10视频 | 用户回访率 | 推荐→点击→完播 |
| SaaS | 各行业Top10客户 | 续费率 | 试用→激活→付费 |
| 金融 | 各基金Top10收益 | 复投率 | 浏览→申购→持有 |

---

## 七、当日小结

> **今天最重要的一个认知**：TopN、复购率、漏斗——三道题覆盖了分析师日常SQL工作的70%。它们的共同模式："先GROUP BY到分析粒度，再用窗口函数或CASE WHEN计算目标指标"。TopN的关键选择是用ROW_NUMBER还是RANK（影响预算），复购率的核心是COUNT(CASE WHEN order_cnt>=2)，漏斗的精髓是要明确"什么粒度"（天级/会话级/事件级）。

**三条金句带回家**：
1. TopN的核心选择：ROW_NUMBER严格N人（控预算），RANK公平但可能超N人，DENSE_RANK连续但不跳号
2. 复购率 = COUNT(CASE WHEN 消费次数>=2 THEN 1 END) / COUNT(*)，按维度拆解就是加GROUP BY列
3. 漏斗的核心问题不是"怎么写SQL"，而是"什么粒度"——天级/会话级/事件级，精度差异巨大

### 课后拓展

1. **练习**：把今天的三道题整合到一个综合月报脚本中——用CTE组织逻辑，生成一份可以直接发给运营的报表

2. **思考**：如果订单表有1亿行，TopN用窗口函数性能如何？有没有替代方案？（提示：使用LATERAL VIEW或预聚合）

3. **预习**：明天是周复习——用今天学的三道经典SQL完成综合任务

4. **面试准备**：背熟以下问题——"如何取每组TopN？""复购率怎么算？""漏斗分析的SQL怎么写？""这三个指标如何综合评估用户质量？"

---

*03analyst_thinking · Week 5 Day 6 · 2026-05-21*